In [1]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options as ChromeOptions
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import requests
import pandas as pd
from datetime import date, timedelta
from functools import reduce
import os
from shutil import which
from bs4 import BeautifulSoup

chromedriver_autoinstaller.install()

'c:\\Python310\\lib\\site-packages\\chromedriver_autoinstaller\\155\\chromedriver.exe'

In [2]:
def _setup_chrome_options():
    """Set up Chrome options for local Selenium execution."""
    options = ChromeOptions()
    
    # Keep Chrome visible while running locally.
    # Uncomment this when you want headless execution again.
    # options.add_argument("--headless=new")
    
    options.add_argument("--start-maximized")
    
    return options

In [3]:
# url = "https://www.rotowire.com/hockey/nhl-lineups.php"
url = "https://www.rotowire.com/hockey/nhl-lineups.php?date=tomorrow"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Daily Lineups | RotoWire
Current URL: https://www.rotowire.com/hockey/nhl-lineups.php?date=tomorrow


In [4]:
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CLASS_NAME, "lineups"))
)

print("Lineup container found!")

Lineup container found!


In [5]:
matchups = []

lineups = (
    driver.find_element(By.CLASS_NAME, "lineups")
    .find_elements(By.CLASS_NAME, "is-nhl")
)

print("Number of lineups found:", len(lineups))

Number of lineups found: 17


In [6]:
for lineup in lineups[:-1]:
    goalies = lineup.find_elements(
        By.CLASS_NAME,
        "lineup__player-highlight-name"
    )

    away_goalie = goalies[0].text.strip().split()[-1]
    home_goalie = goalies[1].text.strip().split()[-1]

    teams = lineup.find_element(
        By.CLASS_NAME,
        "lineup__matchup"
    ).find_elements(By.TAG_NAME, "a")

    if len(teams) >= 2:
        away_team = teams[0].text.split(" (")[0]
        home_team = teams[1].text.split(" (")[0]

        matchups.append(
            (away_team, home_team, away_goalie, home_goalie)
        )

matchups

[('Devils', 'Red Wings', 'Allen', 'Gibson'),
 ('Panthers', 'Blue Jackets', 'Schmid', 'Greaves'),
 ('Sabres', 'Canadiens', 'Luukkonen', 'Dobes'),
 ('Capitals', 'Hurricanes', 'Thompson', 'Bussi'),
 ('Kraken', 'Flyers', 'Daccord', 'Vladar'),
 ('Lightning', 'Rangers', 'Vasilevskiy', 'Garand'),
 ('Blues', 'Senators', 'Hofer', 'Ersson'),
 ('Canucks', 'Islanders', 'Lankinen', 'Sorokin'),
 ('Penguins', 'Blackhawks', 'Silovs', 'Knight'),
 ('Wild', 'Jets', 'Pickard', 'Skinner'),
 ('Avalanche', 'Stars', 'Blackwood', 'Oettinger'),
 ('Golden Knights', 'Predators', 'Hill', 'Saros'),
 ('Maple Leafs', 'Mammoth', 'Bobrovsky', 'Vejmelka'),
 ('Flames', 'Ducks', 'Cooley', 'Dostal'),
 ('Oilers', 'Kings', 'Levi', 'Kuemper'),
 ('Bruins', 'Sharks', 'Swayman', 'Askarov')]

In [7]:
driver.quit()

In [8]:
url = "https://moneypuck.com/goalies.htm"

options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(20)
driver.set_script_timeout(20)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: NHL Goalie Advanced Stats 2026-2027
Current URL: https://moneypuck.com/goalies.htm


In [9]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [10]:
wait = WebDriverWait(driver, 10)

dropdown = wait.until(
    EC.presence_of_element_located((By.ID, "num_games_type"))
)

select = Select(dropdown)

select.select_by_value("_20")

print("Selected Last 20 Games")

Selected Last 20 Games


In [11]:
wait.until(
    EC.presence_of_element_located((By.ID, "goaliesTable"))
)

print("Goalie table found!")

Goalie table found!


In [12]:
table_html = driver.find_element(
    By.ID,
    "goaliesTable"
).get_attribute("outerHTML")

print("HTML length:", len(table_html))

HTML length: 152152


In [13]:
driver.quit()

In [14]:
soup = BeautifulSoup(table_html, "html.parser")

rows = soup.find_all("tr")[1:]

data = []

for tr in rows:
    tds = tr.find_all("td")

    if len(tds) < 7:
        continue

    try:
        rank = int(tds[0].get_text(strip=True))
        goalie = tds[2].get_text(strip=True).split()[-1]
        gsa = float(tds[6].get_text(strip=True))
    except:
        continue

    data.append({
        "Rank": rank,
        "Goalie": goalie,
        "GSAx": gsa
    })

goalie_df = pd.DataFrame(data)

goalie_df.head(20)

,Rank,Goalie,GSAx
0,1,Oettinger,6.6
1,2,Wallstedt,6.2
2,3,Ullmark,5.0
3,4,Shesterkin,4.9
4,5,Swayman,4.8
5,6,Bobrovsky,4.7
6,7,Sorokin,4.5
7,8,Tarasov,4.0
8,9,Murashov,3.2
9,10,Thompson,3.1


In [15]:
goalie_ranks = dict(
    zip(goalie_df["Goalie"], goalie_df["Rank"])
)

goalie_ranks

{'Oettinger': 1,
 'Wallstedt': 2,
 'Ullmark': 3,
 'Shesterkin': 4,
 'Swayman': 5,
 'Bobrovsky': 6,
 'Sorokin': 7,
 'Tarasov': 8,
 'Murashov': 9,
 'Thompson': 10,
 'Markstrom': 11,
 'Saros': 12,
 'Grubauer': 13,
 'Skinner': 14,
 'Woll': 15,
 'Hofer': 16,
 'Fowler': 17,
 'Vejmelka': 18,
 'Nedeljkovic': 19,
 'Merilinen': 20,
 'Kochetkov': 21,
 'Garand': 22,
 'Stolarz': 23,
 'Husso': 24,
 'Talbot': 25,
 'Hildeby': 26,
 'Daccord': 27,
 'Lindgren': 28,
 'Murray': 29,
 'Varlamov': 30,
 'Allen': 31,
 'Silovs': 32,
 'Ellis': 33,
 'Dostal': 34,
 'DeSmith': 35,
 'Vasilevskiy': 36,
 'Hart': 37,
 'Wedgewood': 38,
 'Askarov': 39,
 'Schmid': 40,
 'Cossa': 41,
 'DiPietro': 42,
 'Ersson': 43,
 'Hill': 44,
 'Stevenson': 45,
 'Knight': 46,
 'Bussi': 47,
 'Levi': 48,
 'Daws': 49,
 'Greaves': 50,
 'Gibson': 51,
 'Soderblom': 52,
 'Kuemper': 53,
 'Forsberg': 54,
 'Jarry': 55,
 'Vladar': 56,
 'Binnington': 57,
 'Dobes': 58,
 'Blackwood': 59,
 'Cooley': 60,
 'Luukkonen': 61,
 'Wolf': 62,
 'Lankinen': 63}

In [16]:
url = "https://www.nextteamup.com/nhl"

headers = {
    "User-Agent": "Mozilla/5.0"
}

resp = requests.get(
    url,
    headers=headers,
    timeout=15
)

resp.raise_for_status()

print("Status:", resp.status_code)
print("Response length:", len(resp.text))

Status: 200
Response length: 59315


In [17]:
soup = BeautifulSoup(resp.text, "html.parser")

streaks = {}

rows = soup.select("table tbody tr")

print("Rows found:", len(rows))

Rows found: 32


In [18]:
soup = BeautifulSoup(resp.text, "html.parser")

streaks = {}

rows = soup.select("table tbody tr")

print("Rows found:", len(rows))

Rows found: 32


In [19]:
today = date.today()
start = today - timedelta(days=30)

url = (
    "https://www.naturalstattrick.com/teamtable.php"
    f"?fromseason=20252026"
    f"&thruseason=20252026"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=n"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print(url)

https://www.naturalstattrick.com/teamtable.php?fromseason=20252026&thruseason=20252026&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-13&td=2026-10-13


In [20]:
today = date.today()
start = today - timedelta(days=30)

url = (
    "https://www.naturalstattrick.com/teamtable.php"
    f"?fromseason=20262027"
    f"&thruseason=20262027"
    f"&stype=2"
    f"&sit=5v5"
    f"&score=all"
    f"&rate=n"
    f"&team=all"
    f"&loc=B"
    f"&gpf=410"
    f"&fd={start}"
    f"&td={today}"
)

print(url)

https://www.naturalstattrick.com/teamtable.php?fromseason=20262027&thruseason=20262027&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-13&td=2026-10-13


In [21]:
options = _setup_chrome_options()
driver = webdriver.Chrome(options=options)

driver.set_page_load_timeout(30)
driver.set_script_timeout(30)

driver.get(url)

print("Page title:", driver.title)
print("Current URL:", driver.current_url)

Page title: Just a moment...
Current URL: https://www.naturalstattrick.com/teamtable.php?fromseason=20262027&thruseason=20262027&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-13&td=2026-10-13


In [22]:
html = driver.page_source

print("Page title:", driver.title)
print("Current URL:", driver.current_url)
print("HTML length:", len(html))
print("Contains table:", "<table" in html)
print("Contains Cloudflare:", "Cloudflare" in html)

Page title: Team Season Totals - Natural Stat Trick
Current URL: https://www.naturalstattrick.com/teamtable.php?fromseason=20262027&thruseason=20262027&stype=2&sit=5v5&score=all&rate=n&team=all&loc=B&gpf=410&fd=2026-09-13&td=2026-10-13
HTML length: 71218
Contains table: True
Contains Cloudflare: False


In [23]:
df = pd.read_html(
    html,
    flavor="lxml",
    header=0,
    index_col=0,
    na_values=["-"]
)[0]

df.head()

,Team,GP,TOI,W,L,OTL,ROW,Points,Point %,CF,...,LDSA,LDSF%,LDGF,LDGA,LDGF%,LDSH%,LDSV%,SH%,SV%,PDO
1,Florida Panthers,6,261:28,2,2,2,2,6,0.500,247,...,45,51.61,0,1,0.00,0.00,97.78,7.14,92.31,0.995
2,Carolina Hurricanes,7,313:21,5,1,1,5,11,0.786,337,...,42,59.62,0,1,0.00,0.00,97.62,10.83,88.29,0.991
3,Montreal Canadiens,5,222:13,2,2,1,1,5,0.500,221,...,41,53.41,2,1,66.67,4.26,97.56,8.33,89.25,0.976
4,Toronto Maple Leafs,6,275:27,3,2,1,3,7,0.583,260,...,52,47.47,3,2,60.00,6.38,96.15,10.00,89.31,0.993
5,New York Rangers,7,338:10,5,2,0,5,10,0.714,269,...,49,48.42,5,2,71.43,10.87,95.92,10.32,93.94,1.043


In [24]:
cf = df.sort_values(by="CF%", ascending=False, ignore_index=True)
cf.index += 1
cf.reset_index(inplace=True)
cf = cf.rename(columns={"index": "CF%"})
cf = cf[["Team", "CF%"]]

gf = df.sort_values(by="GF%", ascending=False, ignore_index=True)
gf.index += 1
gf.reset_index(inplace=True)
gf = gf.rename(columns={"index": "GF%"})
gf = gf[["Team", "GF%"]]

xgf = df.sort_values(by="xGF%", ascending=False, ignore_index=True)
xgf.index += 1
xgf.reset_index(inplace=True)
xgf = xgf.rename(columns={"index": "xGF%"})
xgf = xgf[["Team", "xGF%"]]

hdcf = df.sort_values(by="HDCF%", ascending=False, ignore_index=True)
hdcf.index += 1
hdcf.reset_index(inplace=True)
hdcf = hdcf.rename(columns={"index": "HDCF%"})
hdcf = hdcf[["Team", "HDCF%"]]

sh = df.sort_values(by="SH%", ascending=False, ignore_index=True)
sh.index += 1
sh.reset_index(inplace=True)
sh = sh.rename(columns={"index": "SH%"})
sh = sh[["Team", "SH%"]]

In [25]:
cf.head()

,Team,CF%,CF%
0,New Jersey Devils,1,62.89
1,Carolina Hurricanes,2,60.07
2,Utah Mammoth,3,57.47
3,Florida Panthers,4,56.01
4,Buffalo Sabres,5,55.81


In [26]:
dfs = [cf, gf, xgf, hdcf, sh]

final_df = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on="Team",
        how="outer"
    ),
    dfs
)

final_df

,Team,CF%,CF%,GF%,GF%,xGF%,xGF%,HDCF%,HDCF%,SH%,SH%
0,New Jersey Devils,1,62.89,27,37.50,3,61.34,7,58.51,32,4.38
1,Carolina Hurricanes,2,60.07,12,56.67,1,63.37,5,59.20,9,10.83
2,Utah Mammoth,3,57.47,11,57.14,2,63.25,4,60.66,19,8.39
3,Florida Panthers,4,56.01,18,50.00,13,53.08,12,54.41,27,7.14
4,Buffalo Sabres,5,55.81,24,43.48,6,57.68,10,54.84,23,7.94
5,Vegas Golden Knights,6,55.78,16,51.85,4,58.89,6,58.87,22,8.19
6,Tampa Bay Lightning,7,55.58,9,57.89,5,57.73,2,61.11,16,9.73
7,Detroit Red Wings,8,55.32,20,46.15,7,57.06,1,63.10,29,4.96
8,Montreal Canadiens,9,54.57,23,44.44,17,51.71,16,51.81,21,8.33
9,Calgary Flames,10,54.37,31,21.43,8,53.96,21,47.79,30,4.88


In [27]:
def normalize_team(name):
    return (
        name.lower()
        .replace(".", "")
        .replace("é", "e")
        .replace(" ", "")
    )

In [28]:
normalized_streaks = {
    normalize_team(team): streak
    for team, streak in streaks.items()
}

normalized_streaks

{}

In [29]:
res = pd.DataFrame()

for away, home, away_goalie, home_goalie in matchups:

    away_df = final_df[
        final_df["Team"].str.contains(
            away,
            case=False,
            na=False
        )
    ]

    home_df = final_df[
        final_df["Team"].str.contains(
            home,
            case=False,
            na=False
        )
    ]

    away_rank = goalie_ranks.get(away_goalie)
    home_rank = goalie_ranks.get(home_goalie)

    away_df = away_df.assign(
        Goalie=away_goalie,
        Goalie_Rank=away_rank,
        Streak=away_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    home_df = home_df.assign(
        Goalie=home_goalie,
        Goalie_Rank=home_rank,
        Streak=home_df["Team"].apply(
            lambda t: normalized_streaks.get(
                normalize_team(t)
            )
        )
    )

    matchup_df = pd.concat(
        [away_df, home_df],
        ignore_index=True
    )

    res = pd.concat(
        [res, matchup_df],
        ignore_index=True
    )

In [30]:
output_file = "../public/result.csv"

res.to_csv(output_file, index=False)

print(f"Saved {len(res)} rows to {output_file}")

Saved 32 rows to ../public/result.csv
